# 04 — Segmentation Results: Qualitative Visualisations

Reads pre-computed per-image metrics from `05_test.ipynb` output and produces:

- **Sample prediction figures** (best / worst / random by Dice) — `outputs/figures/.../fold_k/sample_predictions/`
- **Cross-fold summary tables** — `vis_fold_summary.csv`, `vis_class_summary.csv`
- **Dice & IoU distribution histograms** — `dice_iou_histogram_by_fold.png`
- **Per-class metrics bar chart** (Dice + IoU + Sensitivity) — `per_class_metrics_bar.png`
- **Dice vs IoU scatter** — `dice_vs_iou_scatter.png`
- **Failure-analysis table** — inline display only

Rather than re-running full batched inference (which `05_test.ipynb` already did), this notebook reads
`fold_k_test_per_image.csv`, selects the best / worst / random images, and runs inference **only on those
~9 images per fold** to render the overlay figures.

Reads checkpoints and writes outputs **directly to Drive**.  
Reads images from the **local SSD copy** (fast inference).

**Prerequisites:** run `03_train.ipynb` **and** `05_test.ipynb` before running this notebook.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

## Cell 2 — Bootstrap: Drive + repo

In [ ]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)
print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"REPO_ROOT:  {REPO_ROOT}")

In [ ]:
import os, psutil
print(f"CPU count:   {os.cpu_count()}")
print(f"RAM total:   {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"RAM avail:   {psutil.virtual_memory().available / 1e9:.1f} GB")

## Cell 3 — EXPERIMENT to visualize

Edit `RECIPE`, `DATASET`, and `SPLIT_SCHEME` to match the experiment you trained in `03_train.ipynb`.

Visualization knobs: `FOLDS_TO_VIS`, `N_BEST/WORST/RANDOM_PER_FOLD`, `SHOW_INLINE`.

In [ ]:
import torch
from configs.seg.reference_experiments import get_experiment, REFERENCE_RECIPES

# ── What to visualize ──────────────────────────────────────────────────────
RECIPE       = "03_dicebce"      # one of REFERENCE_RECIPES
DATASET      = "figshare"        # "figshare" | "brats2020"
SPLIT_SCHEME = "image_level"     # "image_level"  = §13 reference reproduction
                                 # "patient_level" = methodologically correct

EXPERIMENT = get_experiment(RECIPE, dataset=DATASET, split_scheme=SPLIT_SCHEME, fold=1)

assert EXPERIMENT["task"] == "segmentation"
assert EXPERIMENT["dataset"] in ("figshare", "brats2020")
assert EXPERIMENT["split_scheme"] in ("image_level", "patient_level")
assert RECIPE in REFERENCE_RECIPES, \
    f"Unknown recipe {RECIPE!r}. Choose from: {REFERENCE_RECIPES}"

# ── Visualization settings ─────────────────────────────────────────────────
FOLDS_TO_VIS      = [1, 2, 3, 4, 5]  # set [1] first to spot-check
N_BEST_PER_FOLD   = 3
N_WORST_PER_FOLD  = 3
N_RANDOM_PER_FOLD = 3
RANDOM_STATE      = 42
SHOW_INLINE       = True   # False suppresses inline display (faster in long loops)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"experiment   : {EXPERIMENT['name']}")
print(f"dataset      : {EXPERIMENT['dataset']}")
print(f"split_scheme : {EXPERIMENT['split_scheme']}")
print(f"model        : {EXPERIMENT['model_name']}")
print(f"folds        : {FOLDS_TO_VIS}")
print(f"device       : {device}")
print(f"\nAvailable recipes: {REFERENCE_RECIPES}")

## Cell 4 — Copy dataset to local SSD

`copy_to_local` auto-detects a zip at `DRIVE_ROOT/data/<dataset>.zip` and extracts it locally (~2–3 min).  
Falls back to `shutil.copytree` when no zip exists.

Checkpoints and output figures/tables are read from / written to Drive directly (small files, OK over FUSE).

In [ ]:
from src.train_utils    import set_global_seed
from src.notebook_setup import copy_to_local

set_global_seed(EXPERIMENT["seed"])
LOCAL_ROOT = copy_to_local(DRIVE_ROOT, datasets=[EXPERIMENT["dataset"]])
print(f"LOCAL_ROOT: {LOCAL_ROOT}")

## Cell 5 — `visualize_fold` function

Reads pre-computed per-image metrics from `05_test` output, selects the best / worst / random
images by Dice score, and runs single-image inference on those ~N images to render the overlay figures.

In [ ]:
import time
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

from src.sg_data_utils import build_eval_transform
from src.sg_test_utils import load_model_from_pt, predict_mask
from src.sg_vis_utils  import show_image_gt_pred_overlay
from src.file_utils    import experiment_paths, experiment_root_paths


@torch.no_grad()
def visualize_fold(fold: int):
    """
    Reads pre-computed per-image metrics from 05_test output, selects
    best / worst / random images, then runs predict_mask only on those
    ~N images (not the full test set).

    Writes to Drive:
        outputs/figures/<task>/<dataset>/<exp>/fold_<k>/sample_predictions/*.png

    Returns the per-image DataFrame (from 05_test output), or None on failure.
    """
    drive_paths = experiment_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
        fold=fold,
    )

    # ── Load pre-computed per-image metrics from 05_test ──────────────────
    per_image_csv = drive_paths["tables"] / f"fold_{fold}_test_per_image.csv"
    if not per_image_csv.exists():
        print(f"  fold {fold}: fold_{fold}_test_per_image.csv not found.")
        print(f"  Run 05_test.ipynb for this experiment first, then re-run this cell.")
        return None

    per_image_df = pd.read_csv(per_image_csv)
    print(f"\n{'='*60}\n  FOLD {fold}  ({len(per_image_df)} test images, metrics from 05_test)\n{'='*60}")
    print(f"  dice        mean={per_image_df['dice'].mean():.4f}  std={per_image_df['dice'].std():.4f}")
    print(f"  iou         mean={per_image_df['iou'].mean():.4f}")
    print(f"  sensitivity mean={per_image_df['sensitivity'].mean():.4f}  "
          f"precision mean={per_image_df['precision'].mean():.4f}")

    # ── Select best / worst / random ──────────────────────────────────────
    best  = per_image_df.nlargest (N_BEST_PER_FOLD,   "dice")
    worst = per_image_df.nsmallest(N_WORST_PER_FOLD,  "dice")
    rng   = per_image_df.sample(
        n=min(N_RANDOM_PER_FOLD, len(per_image_df)), random_state=RANDOM_STATE,
    )
    selected = (
        pd.concat([best, worst, rng])
        .drop_duplicates(subset="image_id")
        .reset_index(drop=True)
    )
    print(f"  selected {len(selected)} images for figures "
          f"({N_BEST_PER_FOLD} best / {N_WORST_PER_FOLD} worst / {N_RANDOM_PER_FOLD} random)")

    # ── Load checkpoint (from Drive) ─────────────────────────────────────
    pt_path = drive_paths["best_model"]
    if not pt_path.exists():
        print(f"  best_model.pt not found at {pt_path} — skipping figures")
        return per_image_df

    model, _ = load_model_from_pt(pt_path=pt_path, device=device)
    eval_tf   = build_eval_transform(
        image_size=EXPERIMENT["image_size"],
        preprocessing=EXPERIMENT["preprocessing"],
    )

    fig_dir = drive_paths["figures"] / "sample_predictions"
    fig_dir.mkdir(parents=True, exist_ok=True)

    # ── Run predict_mask only for selected images ─────────────────────────
    t0 = time.time()
    kinds = {"best": best, "worst": worst, "random": rng}
    img_to_kind_rank = {}
    for kind, subset in kinds.items():
        for rank, (_, row) in enumerate(subset.iterrows(), start=1):
            img_to_kind_rank.setdefault(row["image_id"], []).append((kind, rank))

    for _, row in selected.iterrows():
        img_id   = row["image_id"]
        img_path = LOCAL_ROOT / row["image_path"]
        gt_path  = LOCAL_ROOT / row["mask_path"]

        _, pred_mask_arr = predict_mask(
            model, img_path, eval_tf,
            device=device, threshold=EXPERIMENT["threshold"],
        )
        gt_raw = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
        gt_mask_arr = (gt_raw > 127).astype(np.uint8) if gt_raw is not None else np.zeros_like(pred_mask_arr)

        img_gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img_gray is not None:
            img_gray = cv2.resize(
                img_gray,
                (EXPERIMENT["image_size"], EXPERIMENT["image_size"]),
                interpolation=cv2.INTER_LINEAR,
            )
        else:
            img_gray = np.zeros((EXPERIMENT["image_size"], EXPERIMENT["image_size"]), np.uint8)

        for kind, rank in img_to_kind_rank.get(img_id, [("selected", 1)]):
            show_image_gt_pred_overlay(
                image     = img_gray,
                gt_mask   = gt_mask_arr,
                pred_mask = pred_mask_arr,
                title     = (
                    f"fold {fold} | {kind} #{rank} | id={img_id} "
                    f"| class={row['tumor_class']} | patient={row['patient_id']}"
                ),
                dice      = float(row["dice"]),
                iou       = float(row["iou"]),
                save_path = fig_dir / f"{kind}_{rank:02d}_id{img_id}.png",
                show      = SHOW_INLINE,
            )

    print(f"  figures -> {fig_dir.relative_to(DRIVE_ROOT)}  ({time.time()-t0:.1f}s)")

    # ── Quick failure-mode summary ────────────────────────────────────────
    show_cols = [c for c in ["image_id", "tumor_class", "dice", "iou", "sensitivity",
                              "gt_mask_area_ratio", "pred_mask_area_ratio"]
                 if c in per_image_df.columns]
    worst5 = per_image_df.nsmallest(5, "dice")[show_cols].round(4)
    print(f"\n  Bottom 5 by Dice (fold {fold}):")
    print(worst5.to_string(index=False))

    return per_image_df


print("visualize_fold defined.")

## Cell 6 — Run visualizations

Iterates over `FOLDS_TO_VIS`. Each fold: loads checkpoint, runs inference, saves figures + table.

In [ ]:
fold_dfs = {}
for fold in FOLDS_TO_VIS:
    fold_dfs[fold] = visualize_fold(fold)

ok_folds = [k for k, v in fold_dfs.items() if v is not None]
print(f"\nDone. {len(ok_folds)}/{len(FOLDS_TO_VIS)} folds visualized: {ok_folds}")

## Cell 7 — Cross-fold summary tables

Aggregates per-fold and per-class Dice / IoU / Sensitivity / Precision across all visualized folds.
Flat column names (`<metric>_mean`, `<metric>_std`) make the CSVs directly readable as tidy data.

Writes two CSVs to `outputs/tables/<task>/<dataset>/<exp>/`:
- `vis_fold_summary.csv` — one row per fold
- `vis_class_summary.csv` — one row per tumor class (pooled across folds)

In [ ]:
from IPython.display import display

all_dfs = [df.assign(fold=fold) for fold, df in fold_dfs.items() if df is not None]

if not all_dfs:
    print("No fold results to summarize.")
else:
    combined = pd.concat(all_dfs, ignore_index=True)

    # Only include metrics that actually exist in this dataset's per-image CSV.
    metric_cols = [c for c in ["dice", "iou", "sensitivity", "precision"]
                   if c in combined.columns]

    # ── Per-fold summary (flat columns, no multi-level headers) ──────────
    fold_agg = {"n_images": ("dice", "count")}
    for m in metric_cols:
        fold_agg[f"{m}_mean"] = (m, "mean")
        fold_agg[f"{m}_std"]  = (m, "std")

    fold_summary = (
        combined.groupby("fold", as_index=False)
        .agg(**fold_agg)
        .round(4)
    )
    fold_summary.insert(0, "experiment", EXPERIMENT["name"])

    # ── Per-class summary (pooled across folds) ──────────────────────────
    class_agg = {"n_images": ("dice", "count")}
    for m in metric_cols:
        class_agg[f"{m}_mean"] = (m, "mean")
        class_agg[f"{m}_std"]  = (m, "std")

    class_summary = (
        combined.groupby("tumor_class", as_index=False)
        .agg(**class_agg)
        .round(4)
    )
    class_summary.insert(0, "experiment", EXPERIMENT["name"])

    print("Per-fold metrics:")
    display(fold_summary)
    print("\nPer-class metrics (pooled across folds):")
    display(class_summary)

    root_paths = experiment_root_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
    )
    out_dir = root_paths["tables"]
    fold_summary.to_csv (out_dir / "vis_fold_summary.csv",  index=False)
    class_summary.to_csv(out_dir / "vis_class_summary.csv", index=False)
    print(f"\nSaved 2 CSVs -> {out_dir.relative_to(DRIVE_ROOT)}")

## Cell 8 — Dice & IoU distribution histograms

Per-fold distribution of Dice (top row) and IoU (bottom row).  
Saved as `outputs/figures/<task>/<dataset>/<exp>/dice_iou_histogram_by_fold.png`.

In [ ]:
import matplotlib.pyplot as plt

ok_list = [(fold, df) for fold, df in fold_dfs.items() if df is not None]

if not ok_list:
    print("No fold results to plot.")
else:
    n = len(ok_list)
    metrics_to_plot = [
        ("dice", "#2196F3", "Dice"),
        ("iou",  "#4CAF50", "IoU"),
    ]
    n_rows = len(metrics_to_plot)

    fig, axes = plt.subplots(n_rows, n, figsize=(4 * n, 3.5 * n_rows), sharey="row")
    if n == 1:
        axes = axes.reshape(n_rows, 1)
    if n_rows == 1:
        axes = axes.reshape(1, n)

    for col, (fold, df) in enumerate(ok_list):
        for row_idx, (metric, color, label) in enumerate(metrics_to_plot):
            ax  = axes[row_idx, col]
            vals = df[metric].values if metric in df.columns else None
            if vals is None:
                ax.set_visible(False)
                continue
            ax.hist(vals, bins=20, range=(0, 1),
                    color=color, edgecolor="white", alpha=0.85)
            ax.axvline(vals.mean(), linestyle="--", color="#f44336",
                       label=f"mean={vals.mean():.3f}")
            ax.set_xlabel(label)
            if col == 0:
                ax.set_ylabel("images (count)")
            if row_idx == 0:
                ax.set_title(f"fold {fold}")
            ax.legend(fontsize=9)
            ax.grid(axis="y", alpha=0.3)

    fig.suptitle(
        f"{EXPERIMENT['name']} · {EXPERIMENT['dataset']} "
        f"— Dice (top) / IoU (bottom) distribution by fold"
    )
    fig.tight_layout()

    root_paths = experiment_root_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
    )
    out_png = root_paths["figures"] / "dice_iou_histogram_by_fold.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"saved {out_png.relative_to(DRIVE_ROOT)}")

## Cell 9 — Per-class metrics bar chart

Mean ± std Dice / IoU / Sensitivity per tumor class, grouped bars per class.  
Error bars reflect cross-fold variance (one mean per fold per class → honest uncertainty
rather than inflated image-level variance).  
Works for brats2020 (single class: glioma) and figshare (3 classes) without code changes.  
Saved as `outputs/figures/<task>/<dataset>/<exp>/per_class_metrics_bar.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_dfs_c = [df.assign(fold=fold) for fold, df in fold_dfs.items() if df is not None]

if not all_dfs_c:
    print("No fold results to plot.")
else:
    combined = pd.concat(all_dfs_c, ignore_index=True)
    metric_cols = [c for c in ["dice", "iou", "sensitivity"] if c in combined.columns]

    # One mean per (tumor_class, fold) → summarize across folds for honest error bars.
    pfc = (
        combined.groupby(["tumor_class", "fold"])[metric_cols]
        .mean()
        .reset_index()
    )

    class_rows = []
    for cls, g in pfc.groupby("tumor_class"):
        row = {"tumor_class": cls}
        for m in metric_cols:
            row[f"{m}_mean"] = g[m].mean()
            row[f"{m}_std"]  = g[m].std(ddof=1) if len(g) > 1 else 0.0
        class_rows.append(row)
    stats = (
        pd.DataFrame(class_rows)
        .sort_values("tumor_class")
        .reset_index(drop=True)
    )

    classes = stats["tumor_class"].tolist()
    n_cls   = len(classes)
    n_met   = len(metric_cols)
    x       = np.arange(n_cls)
    width   = 0.75 / n_met
    offsets = np.linspace(-(n_met - 1) * width / 2, (n_met - 1) * width / 2, n_met)
    palette = ["#2196F3", "#4CAF50", "#FF9800"]

    fig, ax = plt.subplots(figsize=(max(6, n_cls * 2.5 + 1), 5))
    for i, (metric, color) in enumerate(zip(metric_cols, palette)):
        means = stats[f"{metric}_mean"].values
        stds  = stats[f"{metric}_std"].values
        bars  = ax.bar(
            x + offsets[i], means, width,
            yerr=stds, capsize=5, color=color, alpha=0.85, label=metric,
            error_kw={"elinewidth": 1.5, "capthick": 1.5},
        )
        for bar, m in zip(bars, means):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.018,
                f"{m:.3f}", ha="center", va="bottom", fontsize=9,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(classes, fontsize=11)
    ax.set_ylabel("Score  (mean ± std, cross-fold)")
    ax.set_ylim(0, 1.15)
    ax.set_title(
        f"{EXPERIMENT['name']}  ·  {EXPERIMENT['dataset']} "
        f"— Metrics by tumor class  (n={len(ok_folds)} folds)"
    )
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=9)
    fig.tight_layout()

    root_paths = experiment_root_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
    )
    out_png = root_paths["figures"] / "per_class_metrics_bar.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"saved {out_png.relative_to(DRIVE_ROOT)}")

## Cell 10 — Dice vs IoU scatter

Per-image scatter of Dice vs IoU, colored by fold.  
The dashed curve shows the theoretical relationship IoU = Dice / (2 − Dice); points should
cluster tightly on it. Deviations indicate images where the model severely over- or under-segments
relative to its overlap score.  
Saved as `outputs/figures/<task>/<dataset>/<exp>/dice_vs_iou_scatter.png`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

all_dfs_sc = [df.assign(fold=fold) for fold, df in fold_dfs.items() if df is not None]

if not all_dfs_sc:
    print("No fold results to plot.")
else:
    combined  = pd.concat(all_dfs_sc, ignore_index=True)
    folds_uniq = sorted(combined["fold"].unique())
    pal = [cm.get_cmap("tab10")(i) for i in range(len(folds_uniq))]

    fig, ax = plt.subplots(figsize=(6, 6))
    for fold, color in zip(folds_uniq, pal):
        sub = combined[combined["fold"] == fold]
        ax.scatter(sub["dice"], sub["iou"],
                   color=color, s=8, alpha=0.45, label=f"fold {fold}")

    # Theoretical curve: IoU = Dice / (2 − Dice)
    d_line = np.linspace(0, 1, 300)
    i_line = d_line / (2.0 - d_line)
    ax.plot(d_line, i_line, "k--", lw=1.2, alpha=0.5, label="IoU = Dice/(2−Dice)")

    ax.set_xlabel("Dice"); ax.set_ylabel("IoU")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(
        f"{EXPERIMENT['name']}  ·  {EXPERIMENT['dataset']} "
        f"— Dice vs IoU per image"
    )
    ax.legend(fontsize=8, markerscale=2)
    ax.grid(alpha=0.25)
    fig.tight_layout()

    root_paths = experiment_root_paths(
        DRIVE_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
    )
    out_png = root_paths["figures"] / "dice_vs_iou_scatter.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"saved {out_png.relative_to(DRIVE_ROOT)}")

## Cell 11 — Failure analysis

Inline display only — no file saved.

- **Bottom-20 predictions by Dice** across all folds (heat-map colored): identifies the images the model
  struggles most with. `gt_mask_area_ratio` helps spot whether failures correlate with tiny tumors.
- **Per-class failure rate** (fraction with Dice < 0.5): reveals which tumor type is hardest.

In [ ]:
from IPython.display import display

all_dfs_w = [df.assign(fold=fold) for fold, df in fold_dfs.items() if df is not None]

if not all_dfs_w:
    print("No fold results available.")
else:
    combined = pd.concat(all_dfs_w, ignore_index=True)
    fail_threshold = 0.5

    # ── Bottom-N predictions ranked by Dice ──────────────────────────────
    worst_n    = 20
    disp_cols  = [c for c in ["fold", "image_id", "patient_id", "tumor_class",
                               "dice", "iou", "sensitivity",
                               "gt_mask_area_ratio", "pred_mask_area_ratio"]
                  if c in combined.columns]
    worst_df = (
        combined.nsmallest(worst_n, "dice")[disp_cols]
        .round(4)
        .reset_index(drop=True)
    )
    print(f"Bottom {worst_n} predictions by Dice across all folds:")
    display(worst_df.style.background_gradient(subset=["dice"], cmap="RdYlGn"))

    # ── Per-class failure rate ────────────────────────────────────────────
    fail_rows = []
    for cls, g in combined.groupby("tumor_class"):
        fail_rows.append({
            "tumor_class": cls,
            "n_images":    len(g),
            "n_failed":    int((g["dice"] < fail_threshold).sum()),
            "fail_rate":   round(float((g["dice"] < fail_threshold).mean()), 4),
            "dice_mean":   round(float(g["dice"].mean()), 4),
            "dice_std":    round(float(g["dice"].std(ddof=1)), 4) if len(g) > 1 else 0.0,
        })
    fail_by_class = pd.DataFrame(fail_rows)
    print(f"\nPer-class failure rate (Dice < {fail_threshold}):")
    display(fail_by_class)